# Fire: v2.0.0 vs production v1.1.0

Asserts the raster values are bit-identical in the regions where a PROJ version change shows up
(the 8 SE regions where NOAA HARN grids overlap). Reads ~200 MB per region per store: run in-region
(`pixi run start-coiled-notebook`). Only reads stored values, so the kernel's PROJ version doesn't matter.

In [1]:
import icechunk
import xarray as xr

from ocr.risks.fire.grid import ChunkingConfig

OLD = 'output/fire-risk/tensor/production/v1.1.0/ocr.icechunk'
DEV = 'output/fire/v2.0.0/store.icechunk'  # `pixi run fire-all`; or a scratch/<you>/fire/... run
# Differed under PROJ 9.9.0; add more, or grid.valid_region_ids for all of CONUS (slow) if we want
REGIONS = ['y4_x31', 'y4_x32', 'y4_x33', 'y5_x24', 'y5_x25', 'y5_x26', 'y5_x27', 'y5_x34']

In [2]:
def open_store(prefix: str, group: str | None = None) -> xr.Dataset:
    storage = icechunk.s3_storage(
        bucket='carbonplan-ocr', prefix=prefix, region='us-west-2', from_env=True
    )
    store = icechunk.Repository.open(storage).readonly_session('main').store
    return xr.open_zarr(store, group=group, consolidated=False)


old = open_store(OLD)
# dev adds spatial_ref and GeoZarr attrs; assert_equal ignores attrs
dev = open_store(DEV, group='0')[list(old.data_vars)].drop_vars('spatial_ref', errors='ignore')
grid = ChunkingConfig()

In [3]:
for r in REGIONS:
    y, x = grid.region_id_to_latlon_slices(region_id=r)
    xr.testing.assert_equal(old.sel(latitude=y, longitude=x), dev.sel(latitude=y, longitude=x))
    print(r, 'identical')

y4_x31 identical
y4_x32 identical
y4_x33 identical
y5_x24 identical
y5_x25 identical
y5_x26 identical
y5_x27 identical
y5_x34 identical
